# 03 — Local graph methods: LLE, Laplacian eigenmaps (Jia)

Owner: **Jia**. Only Jia edits this notebook.

Template: load the shared data, fit your method, score it with the shared
metrics, compare with the PCA baseline. Name your outputs `03_*` in
`results/` and `figures/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))          # make nldr importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nldr import data, metrics

RESULTS, FIGURES = ROOT / "results", ROOT / "figures"
RESULTS.mkdir(exist_ok=True); FIGURES.mkdir(exist_ok=True)

def save(fig, name):
    """Save a figure as figures/<name>.png and .pdf."""
    for ext in ("png", "pdf"):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=200, bbox_inches="tight")

def plot_rnx(curves, ax, title=None):
    """R_NX(K) against log K. curves: label -> (rnx, J)."""
    for label, (rnx, J) in curves.items():
        ax.semilogx(np.arange(1, len(rnx) + 1), rnx, label=f"{label} (J = {J:.3f})")
    ax.axhline(0, color="0.6", lw=0.8)
    ax.set(xlabel="$K$ (log scale)", ylabel="$R_{NX}(K)$", title=title)
    ax.legend(fontsize=8)

In [ ]:
X, y = data.load_mnist()
s = data.make_splits()[0]               # split 0: indices s.tr, s.val, s.te
Xte, yte = X[s.te], y[s.te]
rho_te = metrics.rank_matrix(Xte)       # input ranks, reuse for every embedding
baseline = pd.read_csv(RESULTS / "baseline_pca_split0.csv", index_col="q")  # from 01_baselines
baseline

## Your method

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding
Y = LocallyLinearEmbedding(n_neighbors=10, n_components=2).fit_transform(Xte)

In [ ]:
res = metrics.evaluate(Xte, Y, yte, rho=rho_te)
rnx = res.pop("rnx")
print(f"J = {res['J']:.4f}   (PCA, q=2: {baseline.loc[2, 'J']:.4f})")

fig, ax = plt.subplots(figsize=(6, 4))
plot_rnx({"my method": (rnx, res["J"])}, ax, title="split 0 (test sample)")
save(fig, "03_example_rnx"); plt.show()